In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))  # Lists available GPUs



[]


In [2]:
import cv2
import tifffile
import numpy as np

from skimage.transform import resize
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as l1_loss
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import mean_squared_error as mse

from scipy.ndimage import gaussian_gradient_magnitude
from scipy.stats import entropy

In [3]:
RGB = "D:/geoinformatics/MajorProject/main/highres_patches/rgbpatchesarcu16/rgbniru16_Clip1_0_0.tif"
PAN = "D:/geoinformatics/MajorProject/main/highres_patches/panpatchesarcu16/WG_PAN_CLIP_BLOCK_0_0.tif"
PANSHARP = "D:/geoinformatics/MajorProject/main/output_tif/bymethods/PanNet/georef/georef_PanNet_0_0.tif"

# # # # Load RGB image
# rgb_image = tifffile.imread(RGB)

# # Load panchromatic image
# pan_image = tifffile.imread(PAN)  

# # Load pansharpened image
# pansharpened_image = tifffile.imread(PANSHARP)  

# print(rgb_image.shape,"&",rgb_image.dtype)
# print(pan_image.shape,"&",pan_image.dtype)
# print(pansharpened_image.shape,"&",pansharpened_image.dtype)

In [6]:
# # # Load RGB image
rgb_image = tifffile.imread(RGB)

# Load panchromatic image
pan_image = tifffile.imread(PAN)  

# Load pansharpened image
pansharpened_image = tifffile.imread(PANSHARP) 

# Convert uint32 to uint8
ms_uint8 = rgb_image.astype(np.uint8)
pan_uint8 = pan_image.astype(np.uint8)
pansharp_uint8 = pansharpened_image.astype(np.uint8)

In [8]:
# Resize the panchromatic image to match the dimensions of the RGB image
pansharp_resized = cv2.resize(pansharp_uint8, (ms_uint8.shape[1], ms_uint8.shape[0]))

# Convert the resized panchromatic image to grayscale
pansharp_gray = cv2.cvtColor(pansharp_resized, cv2.COLOR_BGR2GRAY)
rgb_gray = cv2.cvtColor(ms_uint8,cv2.COLOR_BGR2GRAY)
print(pansharp_resized.shape)
print(ms_uint8.shape)

(512, 512, 4)
(512, 512, 4)


In [9]:
# Now you can calculate the SSIM
ssim_value, _ = ssim(rgb_gray, pansharp_gray, full=True)

# Print the results
print(f"SSIM: {ssim_value}")

SSIM: 0.008418898859522765


In [10]:
# Function to calculate average gradient magnitude
def average_gradient(image):
    gradient_magnitude = gaussian_gradient_magnitude(image, sigma=1)
    return np.mean(gradient_magnitude)


In [11]:
avg_gradient_rgb = average_gradient(rgb_gray)
print("Average Gradient for RGB:", avg_gradient_rgb)

avg_gradient_pansharpened = average_gradient(pansharp_gray)
print("Average Gradient for pansharp:", avg_gradient_pansharpened)

Average Gradient for RGB: 9.8792724609375
Average Gradient for pansharp: 6.248584747314453


In [12]:
# Function to calculate spatial frequency
def spatial_frequency(image):
    return np.mean(np.abs(np.fft.fft2(image)))

In [13]:
spatial_freq_rgb = spatial_frequency(rgb_gray)
print("Spatial Frequency for RGB:", spatial_freq_rgb)

spatial_freq_pansharpened = spatial_frequency(pansharp_gray)
print("Spatial Frequency for pansharp:", spatial_freq_pansharpened)

Spatial Frequency for RGB: 22548.827479617783
Spatial Frequency for pansharp: 2993.490438874447


In [14]:
# Function to calculate standard deviation
def standard_deviation(image):
    return np.std(image)

In [15]:
std_dev_rgb = standard_deviation(rgb_gray)
print("Standard Deviation for RGB:", std_dev_rgb)

std_dev_pansharpened = standard_deviation(pansharp_gray)
print("Standard Deviation for pansharp:", std_dev_pansharpened)

Standard Deviation for RGB: 49.593265522432525
Standard Deviation for pansharp: 17.222393111623738


In [16]:
# Function to calculate overall information loss
def information_loss(original_image, modified_image):
    return entropy(original_image) - entropy(modified_image)

In [18]:
pan_resized2rgb = resize(panchromatic_uint8, rgb_gray.shape, anti_aliasing=True)

pansharporignal_gray = cv2.cvtColor(pansharp_uint8, cv2.COLOR_BGR2GRAY)

In [19]:
info_loss_rgb_pan = information_loss(rgb_gray, pan_resized2rgb)
print("Information Loss (RGB vs Pan):", info_loss_rgb_pan)

info_loss_pansharpened_rgb = information_loss(pansharp_gray, rgb_gray)
print("Information Loss (Pansharpened vs RGB):", info_loss_pansharpened_rgb)

info_loss_pansharpened_pan = information_loss(pansharporignal_gray, panchromatic_uint8)
print("Information Loss (Pansharpened vs Pan):", info_loss_pansharpened_pan)

Information Loss (RGB vs Pan): [-0.03600135 -0.04369029 -0.04376757 -0.04101443 -0.05311374 -0.04808067
 -0.04532688 -0.04972286 -0.04575823 -0.05557189 -0.05403216 -0.05923489
 -0.0481501  -0.04034546 -0.04606618 -0.03578634 -0.04913725 -0.04102785
 -0.04998491 -0.03930376 -0.04559189 -0.0464238  -0.04007119 -0.05188645
 -0.0453694  -0.03903879 -0.04601803 -0.04939044 -0.04435738 -0.04874981
 -0.04026041 -0.04668534 -0.04556825 -0.05221338 -0.05335574 -0.05603348
 -0.05151535 -0.05144042 -0.05012576 -0.0540896  -0.05181942 -0.06012209
 -0.04126147 -0.04278512 -0.05009856 -0.04419825 -0.05330329 -0.05739673
 -0.04551946 -0.05018221 -0.05491655 -0.05186385 -0.04333591 -0.04213233
 -0.04891161 -0.03960591 -0.04466936 -0.05163121 -0.04726116 -0.04828395
 -0.03845447 -0.03777232 -0.04586601 -0.04345366 -0.0582715  -0.057781
 -0.04288774 -0.05275756 -0.05597946 -0.04291869 -0.04554743 -0.04603959
 -0.04697515 -0.05385451 -0.04933058 -0.04768874 -0.04350617 -0.04268042
 -0.04463578 -0.057710

In [20]:
def calculate_aad(difference_image):
    """Calculates the Average Absolute Difference (AAD) between two images.
    Args:
    difference_image: A NumPy array representing the pixel-wise difference between two images.
    Returns:
    The AAD value.
    """
    absolute_differences = np.abs(difference_image)
    average_abs_diff = np.mean(absolute_differences)
    return average_abs_diff

In [21]:
aad_rgb_pan = calculate_aad(info_loss_rgb_pan)
aad_pan_sharpened_rgb = calculate_aad(info_loss_pansharpened_rgb)
aad_pan_sharpened_pan = calculate_aad(info_loss_pansharpened_pan)

print("Average Absolute Difference (AAD):")
print("RGB vs Pan:", aad_rgb_pan)
print("Pansharpened vs RGB:", aad_pan_sharpened_rgb)
print("Pansharpened vs Pan:", aad_pan_sharpened_pan)

Average Absolute Difference (AAD):
RGB vs Pan: 0.047143057464306345
Pansharpened vs RGB: 0.10223057294343565
Pansharpened vs Pan: 0.027402110534878746


In [22]:
print(aad_pan_sharpened_rgb + aad_pan_sharpened_pan)

0.12963268347831441


In [23]:
# Function to calculate Universal Image Quality Index (UIQI)
def calculate_uiqi(reference_image, test_image):
    """Calculates the Universal Image Quality Index (Q) between two images.
    Args:
    reference_image: The reference image (assumed to be higher quality).
    test_image: The test image (whose quality is being measured).
    Returns:
    The Q-index value (between -1 and 1, with 1 indicating perfect quality).
    """ 
    
    # Ensure images have the same data type (e.g., float32) for calculations
    reference_image = reference_image.astype(np.float32)
    test_image = test_image.astype(np.float32)
    
    # Calculate the mean and standard deviation for each image
    mu1 = np.mean(reference_image)
    mu2 = np.mean(test_image)
    sigma1_sq = np.var(reference_image)
    sigma2_sq = np.var(test_image)
    variance_threshold = 1e-6
    
    # Check for near-zero variance in both images
    if sigma1_sq < variance_threshold and sigma2_sq < variance_threshold:
        print("WARNING: Both images have very low variance. UIQI calculation skipped.")
        return None  # Indicate that UIQI is not meaningful


    # Calculate covariance
    covariance = np.mean((reference_image - mu1) * (test_image - mu2))
    print("covariance is:", covariance)
    

    # Define constants
    window_size = 11  # Recommended window size (can be adjusted based on image size)
    c1 = (0.01 * sigma1_sq) ** 2
    c2 = (0.01 * sigma2_sq) ** 2


    # Calculate luminance component with handling for zero denominator
    max_intensity = np.min([np.max(reference_image), np.max(test_image)])
    l_comp = np.where(max_intensity < 255, np.abs(mu1 - mu2) / (255 - max_intensity), 1.0)
    print("luminanace component is:", l_comp)

    # Calculate contrast component
    c_comp = np.max([sigma1_sq, sigma2_sq]) / (sigma1_sq + sigma2_sq + c2)
    print("contrast component is:", c_comp)

    # Calculate correlation component
    numer = covariance + c1
    denom = np.sqrt((sigma1_sq + c1) * (sigma2_sq + c2))
    corr_comp = np.where(denom > 1e-10, np.abs(numer) / denom, 1.0)
    print("correlation component is:", corr_comp)

    # Calculate the Q-index
    q = 4 * corr_comp * l_comp * c_comp

    # Clip Q to the valid range (-1, 1)
    q = np.clip(q, -1, 1)

    return q


In [24]:
rgb_gray_2pansharp = resize(rgb_gray, pansharporignal_gray.shape, anti_aliasing=True).astype(np.uint8)

In [25]:
uiqi_rgb_pan = calculate_uiqi(rgb_gray, pan_resized2rgb)
print("UIQI (RGB vs Pan):", uiqi_rgb_pan)

uiqi_pansharpened_rgb = calculate_uiqi(pansharporignal_gray, rgb_gray_2pansharp)
print("UIQI (Pansharpened vs RGB):", uiqi_pansharpened_rgb)

uiqi_pansharpened_pan = calculate_uiqi(pansharporignal_gray, panchromatic_uint8)
print("UIQI (Pansharpened vs Pan):", uiqi_pansharpened_pan)

covariance is: -0.07549218
luminanace component is: 0.5008858102554758
contrast component is: 0.9999934485649717
correlation component is: 86.05657492574366
UIQI (RGB vs Pan): 1.0
covariance is: 0.0
luminanace component is: 0.09693192126704198
contrast component is: 1.0
correlation component is: 1.0
UIQI (Pansharpened vs RGB): 0.3877276850681679
covariance is: 2.6578622
luminanace component is: 1.0
contrast component is: 0.6249437518840212
correlation component is: 0.007353404015448457
UIQI (Pansharpened vs Pan): 0.01838185557813354


C:\Users\Monia\AppData\Local\Temp\ipykernel_34164\3763415822.py:51: RuntimeWarning: divide by zero encountered in scalar divide
  corr_comp = np.where(denom > 1e-10, np.abs(numer) / denom, 1.0)
C:\Users\Monia\AppData\Local\Temp\ipykernel_34164\3763415822.py:41: RuntimeWarning: divide by zero encountered in scalar divide
  l_comp = np.where(max_intensity < 255, np.abs(mu1 - mu2) / (255 - max_intensity), 1.0)


In [26]:
def calculate_rmse(original_image, pansharpened_image):
    """Calculates the Root Mean Squared Error (RMSE) between two images.
    Args:
    original_image: The original image.
    pansharpened_image: The pansharpened image.
    Returns:
    The RMSE value.
    """
    # Ensure images have the same shape and data type
    if original_image.shape != pansharpened_image.shape:
        raise ValueError("Images must have the same shape.")
    original_image = original_image.astype(np.float32)
    pansharpened_image = pansharpened_image.astype(np.float32)

    # Calculate pixel-wise difference
    squared_difference = np.square(original_image - pansharpened_image)

    # Mean over all pixels
    mean_squared_error = np.mean(squared_difference)
    print("Mean Squared Error (MSE):", mean_squared_error)

    # Calculate RMSE (square root of mean squared error)
    rmse = np.sqrt(mean_squared_error)

    return rmse

In [27]:
rmse = calculate_rmse(rgb_uint8, pansharp_resized)
print("Root Mean Squared Error (RMSE):", rmse)

Mean Squared Error (MSE): 13674.537
Root Mean Squared Error (RMSE): 116.93818
